# SIT307 Task 11.1HD — Reproducing and improving stacking-based heart-disease prediction

Walk-through notebook. Run top-to-bottom (**Kernel → Restart & Run All**) and save with outputs before submission.

* Paper: Bhagat, Sharma & Agarwal (2025), *An efficient stacking-based ensemble technique for early heart attack prediction*.
* Part 1 = reproduction (`src/part1_reproduce.py`), Part 2 = proposed LF-DPS (`src/part2_proposed.py`, `src/part2_estimation_validity.py`).

In [ ]:
import sys, json, subprocess, pandas as pd
from pathlib import Path
from IPython.display import Image, display
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC, RES, FIG = ROOT/'src', ROOT/'results', ROOT/'results'/'figures'
sys.path.insert(0, str(SRC))
import common
print('XGB backend:', common.XGB_BACKEND)

## 1. Data audit — duplicates and sentinel missing values

In [ ]:
df = common.load_data()
audit = common.data_audit(df)
pd.Series(audit)

## 2. Part 1 — reproduction (≈3–5 min)
E1 headline split, E2 30 seeds, E3 meta-learners, E4 leakage audit, E5 de-duplicated.

In [ ]:
subprocess.run([sys.executable, 'part1_reproduce.py'], cwd=SRC, check=True)

In [ ]:
print('Paper (reported):'); display(pd.DataFrame(common.PAPER).T)
print('Reproduced, seed 42:'); display(pd.read_csv(RES/'part1_headline.csv', index_col=0)[['Accuracy','Precision','Recall','F1','AUC']].round(4))

In [ ]:
display(Image(FIG/'p1_confusion.png')); display(Image(FIG/'p1_roc.png'))

In [ ]:
print('30-split mean (duplicates kept):'); display(pd.read_csv(RES/'part1_seeds_summary.csv', index_col=0).filter(like='_mean').round(4))
print('Leakage audit:'); display(pd.read_csv(RES/'part1_leakage.csv', index_col=0).round(3))
print('30-split mean (unique patients):'); display(pd.read_csv(RES/'part1_dedup_summary.csv', index_col=0).filter(like='_mean').round(4))
print('Meta-learner sensitivity:'); display(pd.read_csv(RES/'part1_meta_summary.csv', index_col=0).round(4))

In [ ]:
display(Image(FIG/'p1_leakage.png')); display(Image(FIG/'p1_paper_vs_repro.png')); display(Image(FIG/'p1_importance.png'))

## 3. Part 2 — proposed LF-DPS (≈10–40 min depending on CPU cores)
Nested 5×5 repeated CV on 302 unique patients with the ablation ladder A0→A3.

In [ ]:
subprocess.run([sys.executable, 'part2_proposed.py'], cwd=SRC, check=True)

In [ ]:
display(pd.read_csv(RES/'part2_summary.csv', index_col=0).filter(like='_mean').round(4))
display(pd.read_csv(RES/'part2_stats.csv').round(4))
print(json.load(open(RES/'part2_meta.json')))

In [ ]:
for f in ['p2_boxplots.png','p2_roc_calibration.png','p2_selection.png']: display(Image(FIG/f))

## 4. E6 — estimated vs true accuracy on locked external patients

In [ ]:
subprocess.run([sys.executable, 'part2_estimation_validity.py'], cwd=SRC, check=True)
display(pd.read_csv(RES/'part2_validity_summary.csv', index_col=0).round(4))
display(Image(FIG/'p2_validity.png'))

## 5. Rebuild the PDF report from these results

In [ ]:
subprocess.run([sys.executable, 'make_report.py'], cwd=SRC, check=True)